# 01. 환경 설치 확인

모든 패키지가 올바르게 설치되었는지, GPU가 잘 잡히는지 확인합니다.

In [ ]:
# ── 1. PyTorch & CUDA 확인 ──────────────────────────
import torch

print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️ GPU 없음 - CPU 모드로 진행 (매우 느림)")

In [ ]:
# ── 2. 핵심 패키지 확인 ─────────────────────────────
packages = [
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("ultralytics", "ultralytics"),
    ("gradio", "gradio"),
    ("PIL", "Pillow"),
    ("cv2", "opencv"),
    ("google.generativeai", "google-generativeai"),
]

for module, name in packages:
    try:
        m = __import__(module)
        ver = getattr(m, '__version__', 'OK')
        print(f"✅ {name}: {ver}")
    except ImportError:
        print(f"❌ {name}: 미설치")

In [ ]:
# ── 3. 간단한 SD 파이프라인 로드 테스트 ───────────────
# 주의: 처음 실행 시 HuggingFace에서 모델 다운로드됩니다 (수 GB)

from diffusers import StableDiffusionPipeline

print("SD 파이프라인 로딩 중... (처음이면 다운로드 시작)")

pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    safety_checker=None,
)
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

print("✅ SD 파이프라인 로드 완료!")

In [ ]:
# ── 4. 테스트 이미지 생성 ────────────────────────────
import os
from PIL import Image

os.makedirs("../data/results", exist_ok=True)

result = pipe(
    prompt="a fashionable person wearing a white shirt, studio photo",
    num_inference_steps=20,
    generator=torch.Generator("cuda" if torch.cuda.is_available() else "cpu").manual_seed(42),
)

image = result.images[0]
image.save("../data/results/setup_test.png")

# 노트북에서 바로 보기
display(image)
print("✅ 이미지 생성 테스트 완료! 저장: data/results/setup_test.png")

In [ ]:
# ── 5. YOLOv8 테스트 ─────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # nano 버전, 자동 다운로드
print(f"✅ YOLOv8 로드 완료: {model.info()}")

## 환경 설정 완료!

모든 항목이 ✅ 이면 다음 노트북으로 진행하세요:
- `02_masking_demo.ipynb` - 자동 마스킹 데모
- `03_inpainting_demo.ipynb` - Inpainting 데모
- `04_full_pipeline.ipynb` - 전체 파이프라인